# Tugas Praktik — PySpark Fundamentals
## Studi kasus: Analisis harga cryptocurrency

**Tujuan:** melengkapi kode PySpark untuk membaca data, memilih kolom, melakukan filter, membuat kolom turunan, agregasi, sorting, dan join sederhana.

**Petunjuk pengerjaan**
1. Lengkapi bagian `# TODO` pada setiap cell.
2. Jalankan cell secara berurutan.
3. Periksa hasilnya dan tulis interpretasi singkat pada cell jawaban.
4. Jangan mengganti nama kolom kecuali diminta.

**Catatan:** notebook ini memakai data contoh agar dapat dijalankan tanpa API key, Kafka, atau koneksi database.

## 1. Setup Spark

Jika PySpark belum tersedia di environment, instal terlebih dahulu dengan `pip install pyspark`. Pada cluster kelas, gunakan konfigurasi Spark yang sudah disediakan pengajar.

In [1]:
# TODO 1: import SparkSession dari pyspark.sql
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Tugas_PySpark_Fundamentals")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 12:04:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/11/09 12:04:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.2.0


## 2. Membuat DataFrame

Data berikut menirukan snapshot harga beberapa cryptocurrency. Setiap baris adalah satu catatan harga.

In [2]:
from datetime import datetime

data = [
    (1, "bitcoin", 67250.0, 1.8, "2026-10-22 10:00:00"),
    (2, "ethereum", 2580.0, 2.4, "2026-10-22 10:00:00"),
    (3, "solana", 164.5, -0.8, "2026-10-22 10:00:00"),
    (4, "bitcoin", 67410.0, 2.1, "2026-10-22 10:01:00"),
    (5, "ethereum", 2592.0, 2.8, "2026-10-22 10:01:00"),
    (6, "solana", 163.9, -1.1, "2026-10-22 10:01:00"),
    (7, "dogecoin", 0.18, 4.2, "2026-10-22 10:01:00"),
]

columns = ["record_id", "coin", "price_usd", "change_24h_pct", "snapshot_time"]

# TODO 2: buat DataFrame Spark dari data dan columns
df = spark.createDataFrame(data, columns)

df.show(truncate=False)
df.printSchema()

+---------+--------+---------+--------------+-------------------+
|record_id|coin    |price_usd|change_24h_pct|snapshot_time      |
+---------+--------+---------+--------------+-------------------+
|1        |bitcoin |67250.0  |1.8           |2026-10-22 10:00:00|
|2        |ethereum|2580.0   |2.4           |2026-10-22 10:00:00|
|3        |solana  |164.5    |-0.8          |2026-10-22 10:00:00|
|4        |bitcoin |67410.0  |2.1           |2026-10-22 10:01:00|
|5        |ethereum|2592.0   |2.8           |2026-10-22 10:01:00|
|6        |solana  |163.9    |-1.1          |2026-10-22 10:01:00|
|7        |dogecoin|0.18     |4.2           |2026-10-22 10:01:00|
+---------+--------+---------+--------------+-------------------+

root
 |-- record_id: long (nullable = true)
 |-- coin: string (nullable = true)
 |-- price_usd: double (nullable = true)
 |-- change_24h_pct: double (nullable = true)
 |-- snapshot_time: string (nullable = true)



## 3. Memilih kolom (`select`)

Tugas: tampilkan hanya kolom `coin` dan `price_usd`.

In [3]:
# TODO 3: pilih kolom coin dan price_usd
df.select("coin", "price_usd").show()

+--------+---------+
|    coin|price_usd|
+--------+---------+
| bitcoin|  67250.0|
|ethereum|   2580.0|
|  solana|    164.5|
| bitcoin|  67410.0|
|ethereum|   2592.0|
|  solana|    163.9|
|dogecoin|     0.18|
+--------+---------+



## 4. Filter data

Tugas: tampilkan cryptocurrency dengan `price_usd` lebih besar dari 100.

In [4]:
from pyspark.sql.functions import col

# TODO 4: filter harga > 100
df.filter(col("price_usd") > 100).show()

+---------+--------+---------+--------------+-------------------+
|record_id|    coin|price_usd|change_24h_pct|      snapshot_time|
+---------+--------+---------+--------------+-------------------+
|        1| bitcoin|  67250.0|           1.8|2026-10-22 10:00:00|
|        2|ethereum|   2580.0|           2.4|2026-10-22 10:00:00|
|        3|  solana|    164.5|          -0.8|2026-10-22 10:00:00|
|        4| bitcoin|  67410.0|           2.1|2026-10-22 10:01:00|
|        5|ethereum|   2592.0|           2.8|2026-10-22 10:01:00|
|        6|  solana|    163.9|          -1.1|2026-10-22 10:01:00|
+---------+--------+---------+--------------+-------------------+



## 5. Membuat kolom turunan (`withColumn`)

Tugas: buat kolom `price_usd_idr` dengan asumsi kurs latihan 1 USD = Rp16.000.

In [5]:
# TODO 5: buat kolom price_usd_idr = price_usd * 16000
df_idr = df.withColumn("price_usd_idr", col("price_usd") * 16000)

df_idr.select("coin", "price_usd", "price_usd_idr").show()

+--------+---------+-------------+
|    coin|price_usd|price_usd_idr|
+--------+---------+-------------+
| bitcoin|  67250.0|      1.076E9|
|ethereum|   2580.0|      4.128E7|
|  solana|    164.5|    2632000.0|
| bitcoin|  67410.0|    1.07856E9|
|ethereum|   2592.0|     4.1472E7|
|  solana|    163.9|    2622400.0|
|dogecoin|     0.18|       2880.0|
+--------+---------+-------------+



## 6. Menghitung metrik agregasi

Tugas: hitung rata-rata dan harga tertinggi untuk setiap coin. Gunakan `groupBy` dan `agg`.

In [6]:
from pyspark.sql.functions import avg, max as spark_max

# TODO 6: group by coin, lalu hitung avg(price_usd) dan max(price_usd)
coin_summary = (
    df.groupBy("coin")
    .agg(
        avg("price_usd").alias("avg_price_usd"),
        spark_max("price_usd").alias("max_price_usd")
    )
)

coin_summary.show()

+--------+-------------+-------------+
|    coin|avg_price_usd|max_price_usd|
+--------+-------------+-------------+
| bitcoin|      67330.0|      67410.0|
|ethereum|       2586.0|       2592.0|
|  solana|        164.2|        164.5|
|dogecoin|         0.18|         0.18|
+--------+-------------+-------------+



## 7. Sorting

Tugas: urutkan data berdasarkan `change_24h_pct` dari nilai tertinggi ke terendah.

In [7]:
# TODO 7: urutkan descending berdasarkan change_24h_pct
df.orderBy(col("change_24h_pct").desc()).show()

+---------+--------+---------+--------------+-------------------+
|record_id|    coin|price_usd|change_24h_pct|      snapshot_time|
+---------+--------+---------+--------------+-------------------+
|        7|dogecoin|     0.18|           4.2|2026-10-22 10:01:00|
|        5|ethereum|   2592.0|           2.8|2026-10-22 10:01:00|
|        2|ethereum|   2580.0|           2.4|2026-10-22 10:00:00|
|        4| bitcoin|  67410.0|           2.1|2026-10-22 10:01:00|
|        1| bitcoin|  67250.0|           1.8|2026-10-22 10:00:00|
|        3|  solana|    164.5|          -0.8|2026-10-22 10:00:00|
|        6|  solana|    163.9|          -1.1|2026-10-22 10:01:00|
+---------+--------+---------+--------------+-------------------+



## 8. Join dua DataFrame

Kita memiliki informasi kategori untuk setiap coin. Gabungkan dengan data harga menggunakan kolom `coin`.

In [8]:
coin_info = [
    ("bitcoin", "Layer 1"),
    ("ethereum", "Layer 1"),
    ("solana", "Layer 1"),
    ("dogecoin", "Meme coin"),
]
info_df = spark.createDataFrame(coin_info, ["coin", "category"])

# TODO 8: inner join df dengan info_df berdasarkan kolom coin
joined_df = df.join(info_df, on="coin", how="inner")

joined_df.select("coin", "category", "price_usd").show()

+--------+---------+---------+
|    coin| category|price_usd|
+--------+---------+---------+
| bitcoin|  Layer 1|  67250.0|
| bitcoin|  Layer 1|  67410.0|
|dogecoin|Meme coin|     0.18|
|ethereum|  Layer 1|   2580.0|
|ethereum|  Layer 1|   2592.0|
|  solana|  Layer 1|    164.5|
|  solana|  Layer 1|    163.9|
+--------+---------+---------+



## 9. Menulis hasil ke Parquet

Tugas: simpan hasil ringkasan coin ke folder `output/coin_summary`. Mode `overwrite` digunakan agar tugas dapat dijalankan ulang.

In [9]:
# TODO 9: tulis coin_summary ke format parquet dengan mode overwrite
(
    coin_summary.write
    .format("parquet")
    .mode("overwrite")
    .save("output/coin_summary")
)

In [10]:
# Verifikasi hasil yang tersimpan
saved_df = spark.read.parquet("output/coin_summary")
saved_df.show()

+--------+-------------+-------------+
|    coin|avg_price_usd|max_price_usd|
+--------+-------------+-------------+
| bitcoin|      67330.0|      67410.0|
|ethereum|       2586.0|       2592.0|
|  solana|        164.2|        164.5|
|dogecoin|         0.18|         0.18|
+--------+-------------+-------------+



## 10. Interpretasi hasil

Tuliskan jawaban singkat di bawah ini.

1. Coin mana yang memiliki rata-rata harga tertinggi?
2. Mengapa filter dilakukan menggunakan `col("price_usd") > 100`, bukan membandingkan harga sebagai teks?
3. Apa perbedaan `select`, `filter`, dan `withColumn`?
4. Mengapa Parquet sering digunakan untuk menyimpan data analitik?

In [11]:
# Tulis jawaban atau catatan Anda sebagai komentar Python
# Jawaban 1: Bitcoin memiliki rata-rata harga tertinggi, yaitu USD 67.330.
# Jawaban 2: price_usd bertipe numerik sehingga perbandingan > 100 dilakukan berdasarkan nilai, bukan urutan leksikografis teks.
# Jawaban 3: select memilih kolom, filter memilih baris berdasarkan kondisi, dan withColumn menambah atau mengganti kolom.
# Jawaban 4: Parquet bersifat columnar, terkompresi, menyimpan schema, dan memungkinkan Spark membaca hanya kolom yang diperlukan.

## Checklist sebelum dikumpulkan

- [x] Semua cell TODO sudah dilengkapi.
- [x] Semua cell berhasil dijalankan.
- [x] Output join dan agregasi sudah diperiksa.
- [x] Jawaban interpretasi sudah ditulis.
- [x] Notebook disimpan dengan output yang terlihat.

**Pengumpulan:** kumpulkan file `.ipynb` yang sudah dikerjakan.